## **1. Import Library**

In [ ]:
import warnings
import numpy as np
import pandas as pd
import sklearn
import seaborn as sns
import plotly
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# ----- Konstanta -----
RANDOM_STATE = 42
TEST_SIZE = 0.20

PLOTLY_TEMPLATE = "plotly_white"          
FIGURE_WIDTH = 900
FIGURE_HEIGHT = 500

SEABORN_DEEP_PALETTE = [
    "#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3",
    "#937860", "#DA8BC3", "#8C8C8C", "#CCB974", "#64B5CD",
]


def configure_environment() -> None:
    warnings.filterwarnings("ignore")

    pio.templates.default = PLOTLY_TEMPLATE
    px.defaults.template = PLOTLY_TEMPLATE
    px.defaults.color_discrete_sequence = SEABORN_DEEP_PALETTE
    px.defaults.width = FIGURE_WIDTH
    px.defaults.height = FIGURE_HEIGHT

    pd.set_option("display.max_columns", 50)
    pd.set_option("display.width", 120)


def report_versions() -> None:
    print("Seluruh pustaka berhasil diimpor.")
    print(f"Versi pandas       : {pd.__version__}")
    print(f"Versi numpy        : {np.__version__}")
    print(f"Versi scikit-learn : {sklearn.__version__}")
    print(f"Versi plotly       : {plotly.__version__}")


configure_environment()
report_versions()

Seluruh pustaka berhasil diimpor.
Versi pandas       : 2.3.3
Versi numpy        : 2.5.0
Versi scikit-learn : 1.9.0
Versi plotly       : 6.9.0


## **2. Dataset Load**

In [ ]:
COLUMN_NAMES = [
    "checking_status", "duration_months", "credit_history", "purpose",
    "credit_amount", "savings_status", "employment_since", "installment_rate",
    "personal_status_sex", "other_debtors", "residence_since", "property_type",
    "age_years", "other_installment_plans", "housing", "existing_credits",
    "job", "num_dependents", "telephone", "foreign_worker", "target",
]

def resolve_raw_path(filename="credit_scoring_raw.csv"):
    candidates = [
        Path(filename),                      
        Path("..") / filename,                
        Path("..") / "namadataset_raw" / filename,
        Path("namadataset_raw") / filename,
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(
        f"'{filename}' tidak ditemukan. Lokasi yang dicek: "
        + ", ".join(str(c) for c in candidates)
    )

RAW_PATH = resolve_raw_path()
df = pd.read_csv(RAW_PATH, header=None, names=COLUMN_NAMES)

print(f"Dataset dimuat dari: {RAW_PATH}")
print(f"Dimensi            : {df.shape[0]} baris x {df.shape[1]} kolom")
df.head()


Dataset dimuat dari: ..\credit_scoring_raw.csv
Dimensi            : 1000 baris x 21 kolom


,checking_status,duration_months,credit_history,purpose,credit_amount,savings_status,employment_since,installment_rate,personal_status_sex,other_debtors,residence_since,property_type,age_years,other_installment_plans,housing,existing_credits,job,num_dependents,telephone,foreign_worker,target
0,A11,6,A34,A43,1169,A65,A75,4,A93,A101,4,A121,67,A143,A152,2,A173,1,A192,A201,1
1,A12,48,A32,A43,5951,A61,A73,2,A92,A101,2,A121,22,A143,A152,1,A173,1,A191,A201,2
2,A14,12,A34,A46,2096,A61,A74,2,A93,A101,3,A121,49,A143,A152,1,A172,2,A191,A201,1
3,A11,42,A32,A42,7882,A61,A74,2,A93,A103,4,A122,45,A143,A153,1,A173,2,A191,A201,1
4,A11,24,A33,A40,4870,A61,A73,3,A93,A101,4,A124,53,A143,A153,2,A173,2,A191,A201,2


In [ ]:
# -------- Struktur & tipe data --------
df.info()


## **3. Exploratory Data Analysis (EDA)**

In [ ]:
# -------- Kualitas data: nilai kosong & duplikat --------
missing = df.isna().sum()
print("Total nilai kosong :", int(missing.sum()))
print("Total baris duplikat:", int(df.duplicated().sum()))

# ----- Statistik deskriptif -----
df.describe().T


In [ ]:
# -------- Distribusi target -------
target_label = df["target"].map({1: "Good (1)", 2: "Bad (2)"})
counts = target_label.value_counts()
pct = (counts / len(df) * 100).round(1)

fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(x=counts.index, y=counts.values, ax=ax)
for i, (n, p) in enumerate(zip(counts.values, pct.values)):
    ax.text(i, n + 8, f"{n} ({p}%)", ha="center", fontweight="bold")
ax.set_title("Target Distribution — Credit Risk")
ax.set_xlabel("Class"); ax.set_ylabel("Jumlah nasabah")
plt.tight_layout(); plt.show()

In [ ]:
# ------ Distribusi -------
NUMERIC = ["duration_months", "credit_amount", "installment_rate",
           "residence_since", "age_years", "existing_credits", "num_dependents"]

fig, axes = plt.subplots(3, 3, figsize=(14, 10))
for ax, col in zip(axes.ravel(), NUMERIC):
    sns.histplot(df[col], kde=True, ax=ax, bins=25)
    ax.set_title(col)
for ax in axes.ravel()[len(NUMERIC):]:
    ax.set_visible(False)
plt.suptitle("Numerical Distribution", y=1.01, fontsize=14)
plt.tight_layout(); plt.show()


In [ ]:
# ----- Deteksi Outlier -----
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, ["credit_amount", "duration_months", "age_years"]):
    sns.boxplot(y=df[col], ax=ax)
    ax.set_title(col)
plt.suptitle("Boxplot — Outlier Testing", y=1.03, fontsize=13)
plt.tight_layout(); plt.show()


In [ ]:
# ─--- Relasi fitur  -----
key_cats = ["checking_status", "credit_history", "savings_status"]
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, col in zip(axes, key_cats):
    sns.countplot(data=df.assign(Risk=target_label),
                  x=col, hue="Risk", ax=ax)
    ax.set_title(col); ax.tick_params(axis="x", rotation=45)
plt.suptitle("Fitur Kategorikal Kunci vs Credit Risk", y=1.03, fontsize=14)
plt.tight_layout(); plt.show()


In [ ]:
# ---- Matriks korelasi ----
corr_df = df[NUMERIC].copy()
corr_df["is_bad"] = (df["target"] == 2).astype(int) 
corr = corr_df.corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            square=True, linewidths=.5, ax=ax)
ax.set_title("Numerical Correlation")
plt.tight_layout(); plt.show()


## **4. Data Preprocessing**

In [ ]:
# ---- Pemetaan target ke credit risk ----
data = df.copy()
data["credit_risk"] = data["target"].map({1: 0, 2: 1})
data = data.drop(columns=["target"])

print("Credit Risk Distribution (0=good, 1=bad):")
print(data["credit_risk"].value_counts().sort_index().to_dict())


In [ ]:
# ---- Penanganan missing value & duplikat ----
before = len(data)
data = data.drop_duplicates().reset_index(drop=True)
print(f"Baris duplikat dihapus : {before - len(data)}")

assert data.isna().sum().sum() == 0, "Masih terdapat nilai kosong!"
print("Nilai kosong tersisa   :", int(data.isna().sum().sum()))


In [ ]:
# ---- Pemisahan fitur/target & penetapan tipe kolom ----
TARGET = "credit_risk"
categorical_cols = data.select_dtypes(include="object").columns.tolist()
numeric_cols = [c for c in NUMERIC if c in data.columns]

X = data.drop(columns=[TARGET])
y = data[TARGET]

print(f"Fitur numerik    ({len(numeric_cols)}): {numeric_cols}")
print(f"Fitur kategorikal ({len(categorical_cols)}): {categorical_cols}")
print(f"X: {X.shape} | y: {y.shape}")


In [ ]:
# ---- Pemisahan data terstratifikasi ----
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)
print(f"Set latih : {X_train.shape[0]} baris")
print(f"Set uji   : {X_test.shape[0]} baris")
print("Proporsi bad — latih: {:.3f} | uji: {:.3f}".format(
    y_train.mean(), y_test.mean()))


In [ ]:
# ---- Standardisasi + One-Hot Encoding ----
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False),
         categorical_cols),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

# fit HANYA pada set latih, lalu transform kedua set
X_train_arr = preprocessor.fit_transform(X_train)
X_test_arr = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()
X_train_prep = pd.DataFrame(X_train_arr, columns=feature_names, index=X_train.index)
X_test_prep = pd.DataFrame(X_test_arr, columns=feature_names, index=X_test.index)

print(f"Jumlah fitur setelah encoding: {X_train_prep.shape[1]} "
      f"(dari {X_train.shape[1]} fitur asli)")
X_train_prep.head()


In [ ]:
# ---- Rakit data train ready & simpan ---- 
train_ready = X_train_prep.copy()
train_ready[TARGET] = y_train.values
test_ready = X_test_prep.copy()
test_ready[TARGET] = y_test.values

OUT_DIR = Path("credit_scoring_preprocessing")
OUT_DIR.mkdir(exist_ok=True)
train_ready.to_csv(OUT_DIR / "train.csv", index=False)
test_ready.to_csv(OUT_DIR / "test.csv", index=False)

# Validasi akhir data siap latih
assert train_ready.isna().sum().sum() == 0
assert test_ready.isna().sum().sum() == 0
print("Artefak preprocessing tersimpan di:", OUT_DIR.resolve())
print(f"  • train.csv : {train_ready.shape[0]} baris x {train_ready.shape[1]} kolom")
print(f"  • test.csv  : {test_ready.shape[0]} baris x {test_ready.shape[1]} kolom")
print("Validasi: tidak ada nilai kosong pada data siap latih.")
train_ready.head()


---
## Ringkasan

**Rencana konversi (di luar cakupan notebook ini):**
- **Skilled** — mengonversi langkah §5 menjadi `automate_Nama-siswa.py`: satu fungsi
  preprocessing yang mereplikasi tahapan identik dan mengembalikan data siap latih.
- **Advance** — memicu `automate_Nama-siswa.py` melalui **GitHub Actions** setiap
  kali *trigger* terpantik, menghasilkan dataset terproses terbaru secara otomatis.
